<a href="https://colab.research.google.com/github/Sameekshaingole/fraud-detection-federated-learning/blob/main/Federated_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Client Code (Bank side)
import flwr as fl
from sklearn.linear_model import LogisticRegression

class BankClient(fl.client.NumPyClient):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.model = LogisticRegression(max_iter=1000)

    def get_parameters(self, config):
        return [self.model.coef_, self.model.intercept_]

    def set_parameters(self, parameters):
        self.model.coef_ = parameters[0]
        self.model.intercept_ = parameters[1]

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.fit(self.X, self.y)
        return self.get_parameters(config), len(self.X), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        loss = 1 - self.model.score(self.X, self.y)
        return loss, len(self.X), {"accuracy": self.model.score(self.X, self.y)}

In [ ]:
#Create Clients
client1 = BankClient(bank1_X, bank1_y)
client2 = BankClient(bank2_X, bank2_y)
client3 = BankClient(bank3_X, bank3_y)

In [ ]:
#Server Strategy
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    min_fit_clients=3,
    min_available_clients=3,
)

In [ ]:
#Start Federated Training
fl.simulation.start_simulation(
    client_fn=lambda cid: [client1, client2, client3][int(cid)],
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=5),
    strategy=strategy,
)